# Object-oriented programming 🧱

## What you will learn in this course 🧐🧐

You already know functions, data collections, and control flow. These tools work well for small scripts. But they break down when systems grow complex. Python's **Object-Oriented Programming (OOP)** is a way to organize code. You group related data and behavior into reusable components that model real work operations. This approach turns scattered functions and variables into clear systems that scale with your business.

By the end of this course, you will be able to:

- Understand classes as blueprints and objects as instances
- Distinguish instance attributes from class variables
- Implement methods that act on object state
- Apply the constructor mechanism through `__init__`
- Use encapsulation to protect data integrity
- Control attribute access with properties

In [1]:
from typing import List, Dict

## Procedural code vs object-oriented programming

<img src="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D3_Procedural_Object.png"/>

**Procedural code** works fine for small projects. You have a few functions and a few lists. Everything feels under control. But as the system grows, your logic gets scattered. Small updates become fragile. One change can break many places.

Think about a personal reading list. You want to track books: `title`, `author`, `number of pages`, and `how many times you read each one`. A procedural solution uses one list for each piece of data.

Here is the procedural version. Watch how the data is split across many parallel lists.

In [2]:
# Procedural approach: one list per attribute
book_titles = []
book_authors = []
book_pages = []
book_read_count = []

def add_book(title, author, pages):
    book_titles.append(title)
    book_authors.append(author)
    book_pages.append(pages)
    book_read_count.append(0)

def mark_as_read(book_index):
    book_read_count[book_index] += 1

This code has **three serious problems**:

1. You must track the **index** of each book by hand. If a list gets out of order, everything breaks.
2. The data that belongs to one book is **scattered** across four lists. One book is not one thing. It is pieces in four places.
3. Any function can change any list at any time. There is **no protection** against bad values.

> The more books you add, the harder it gets to keep everything consistent.

OOP solves this by **bundling** data and behavior together in one object.

## Classes and objects

A **class** is a blueprint. It defines:

- the **structure** (what data an object holds) 
- the **behavior** (what an object can do).

An **object** is a concrete object made from that blueprint. From one `Book` class, you can create as many book objects as you need. Each object holds its own data.

Here is the smallest possible `Book` class. It has one constructor and one method.

In [3]:
class Book:
    def __init__(self, title, author, pages): # <- constructor method
        # self.title attaches the value to THIS specific book
        self.title = title
        self.author = author
        self.pages = pages
        self.read_count = 0

    def mark_as_read(self):
        self.read_count += 1
        return f'"{self.title}" read {self.read_count} time(s)'

Line by line:

- `class Book:` starts the blueprint. The name uses [PascalCase](https://builtin.com/articles/pascal-case-vs-camel-case#:~:text=camelCase%3A%20Use%20for%20variables%20and,TypeScript%20(React)%20and%20Java.) by convention.
- `def __init__(self, title, author, pages):` is the **constructor**. Python runs it automatically when you create a new book.
- `self` is the object being created. Python passes it to every method without you needing to write it.
- `self.title = title` stores the value on **this specific** book. Each book gets its own copy.
- `self.read_count = 0` sets a starting value. Every new book begins unread.
- `mark_as_read` is a **method**: a function that belongs to the class.

<Note type="important">

Always write `self` as the first parameter of every instance method. Python uses it to know **which** object the method should act on.

</Note>

Now you create two independent book objects from the same class.

In [4]:
# first book
dune = Book("Dune", "Frank Herbert", 412)

# second book
hobbit = Book("The Hobbit", "J.R.R. Tolkien", 310)

# mark them as read
dune.mark_as_read()
dune.mark_as_read()

hobbit.mark_as_read()

print(f"Dune read {dune.read_count} time(s)")
print(f"The Hobbit read {hobbit.read_count} time(s)")

Dune read 2 time(s)
The Hobbit read 1 time(s)


`dune` and `hobbit` are two separate **instances** of the same class. Each one holds its own title, author, pages, and read count. Marking `dune` as read twice does not affect `hobbit`.

> One class, many independent objects. Each object owns its own state.

Compare this to the procedural version. No index tracking. No risk of lists drifting apart. Each book is a single, self-contained unit.

## Attributes

Attributes are variables attached to a class or an object. Python has two kinds:

- **Instance attributes** belong to one specific object. Example: `self.title`. Each book has its own title.
- **Class attributes** belong to the class itself and are **shared** by every object. Example: a minimum page rule that applies to all books.

In [5]:
class Book:
    # Class attribute: shared by all books
    minimum_pages = 1

    def __init__(self, title, pages):
        # Instance attributes: unique to this book
        self.title = title
        self.pages = pages

You create two books. Each has its own `title` and `pages`. But the `minimum_pages` rule is the same for both because it lives on the class.

In [6]:
dune = Book("Dune", 412)
hobbit = Book("The Hobbit", 310)

print(f"Dune pages: {dune.pages}")
print(f"Hobbit pages: {hobbit.pages}")
print(f"Minimum pages (class rule): {Book.minimum_pages}")

Dune pages: 412
Hobbit pages: 310
Minimum pages (class rule): 1


`dune.pages` and `hobbit.pages` return different values because each book has its own instance attribute. `Book.minimum_pages` returns the shared value from the class itself. If you change `Book.minimum_pages = 50`, every book sees the new rule at once.

<Note type="tip">

Use **instance attributes** for data that changes per object (title, author, count). Use **class attributes** for rules, defaults, or counters shared across all objects.

</Note>

## Encapsulation

<Note type="important">

Direct attribute access is dangerous. Any line of code anywhere in your program can assign any value to any attribute. Bad values can corrupt your data silently.

</Note>

**Encapsulation** means the object controls its own data. External code cannot change internal values directly. Every change must pass through a method or property that validates the input.

**Python uses the underscore prefix** (`_read_count`) to signal that an attribute is private. It is a convention, not a lock. It tells other developers: *do not touch this directly*.

First, look at an unprotected class. Anything can be written to it.

In [7]:
class Book:
    def __init__(self, title):
        self.title = title
        self.pages = 0
        self.read_count = 0

book = Book("Dune")
book.pages = -50          # negative pages: nonsense, but accepted
book.read_count = "many"  # wrong type, but accepted
book.colour = "blue"      # a brand new attribute appears silently

Nothing stopped these bad values. The object has no defense. In a real system, a negative page count could break reports, exports, or reading time calculations.

Now look at the protected version. Internal data uses underscores, and a **property** controls access.

In [8]:
class Book:
    def __init__(self, title):
        self.title = title
        self._read_count = 0  # underscore = private

    def mark_as_read(self):
        self._read_count += 1  # only way to change the count

    @property
    def read_count(self):
        return self._read_count  # read-only view

The `@property` decorator turns the `read_count` method into a **read-only attribute**. You can read `book.read_count`, but you cannot assign to it. The only way to change the count is through `mark_as_read()`, which only adds one at a time.

Let's test it.

In [9]:
dune = Book("Dune")
dune.mark_as_read()
print(dune.read_count)  # 1

try:
    dune.read_count = -50
except AttributeError as e:
    print(f"Assignment blocked: {e}")

1
Assignment blocked: property 'read_count' of 'Book' object has no setter


Python refuses the assignment because no setter method exists. The object **owns** its data. External code cannot bypass the rules.

<Note type="important">

Encapsulation is about **trust**. You trust methods to change state correctly. You do not trust external code to respect your rules. Properties enforce this.

</Note>

### Properties with validation

Sometimes you need to allow assignment, but with validation. You add a **setter** that checks the value before storing it.

In [10]:
class Book:
    def __init__(self, title):
        self.title = title
        self._pages = 0

    @property
    def pages(self):
        return self._pages

    @pages.setter
    def pages(self, value):
        if not isinstance(value, int):
            raise TypeError("Pages must be an integer")
        if value < 1:
            raise ValueError("A book must have at least one page")
        self._pages = value

The `@pages.setter` decorator lets you write a method that runs every time someone assigns to `pages`. It checks the type first, then the range. Only valid values reach `self._pages`.

In [11]:
dune = Book("Dune")
dune.pages = 412          # works: passes validation
print(dune.pages)

try:
    dune.pages = "many"   # blocked: wrong type
except TypeError as e:
    print(f"Blocked: {e}")

412
Blocked: Pages must be an integer


From the outside, `dune.pages` still looks like a simple attribute. Inside, Python runs the full validation every time. This is the power of properties: **clean syntax, strong control**.

## Method types

Python has three kinds of methods. Each has a different job.

- **Instance method**: acts on one specific object. Uses `self`. Most methods are this kind.
- **Class method**: acts on class-level data (for example, counting all books ever created). Uses `cls` and the `@classmethod` decorator.
- **Static method**: does not touch any object or class data. It is just a helper function placed inside the class for organization. Uses `@staticmethod`.

In [12]:
class Book:
    total_books = 0  # class attribute: shared counter

    def __init__(self, title):
        self.title = title
        Book.total_books += 1

    # Instance method: needs a specific book
    def describe(self):
        return f"Book: {self.title}"

    # Class method: works on class data
    @classmethod
    def get_count(cls):
        return cls.total_books

    # Static method: pure helper, no self or cls
    @staticmethod
    def is_valid_isbn(isbn):
        return isinstance(isbn, str) and len(isbn) == 13

Each method type is called differently.

In [13]:
dune = Book("Dune")
hobbit = Book("The Hobbit")

# Instance method: called on an object
print(dune.describe())

# Class method: called on the class itself
print(f"Total books: {Book.get_count()}")

# Static method: called on the class, no object needed
print(f"Valid ISBN: {Book.is_valid_isbn('9782070360024')}")

Book: Dune
Total books: 2
Valid ISBN: True


- `dune.describe()` needs a specific book. It reads `self.title`.
- `Book.get_count()` works on the class. It reads `cls.total_books` which is shared across all books.
- `Book.is_valid_isbn('...')` is a pure utility. It does not care about any book or the class. It is inside the `Book` class only for organization.

<Note type="tip">

Use `@classmethod` when the method needs shared data or creates objects in a special way. Use `@staticmethod` for helpers that logically belong to the class but do not need any state.

</Note>

## Special methods

Special methods (also called **dunder methods** because of the double underscores) connect your objects to Python's built-in operations. You define them once, and Python calls them automatically when your object meets common syntax.

| Method | What it controls | Example |
|--------|------------------|---------|
| `__str__` | How the object prints for users | `print(book)` |
| `__repr__` | How the object shows for developers | Console debugging |
| `__eq__` | Equality comparison | `book1 == book2` |
| `__lt__` | Less-than comparison | `sorted(books)` |

In [14]:
class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.author = author
        self.pages = pages

    def __str__(self):
        # Friendly display for users
        return f'"{self.title}" by {self.author}'

    def __repr__(self):
        # Technical display for developers
        return f"Book('{self.title}', '{self.author}', {self.pages})"

    def __eq__(self, other):
        # Two books are equal if title and author match
        if not isinstance(other, Book):
            return False
        same_title = self.title == other.title
        same_author = self.author == other.author
        return same_title and same_author

    def __lt__(self, other):
        # Sort books by number of pages
        return self.pages < other.pages

Each method is called by Python when you use a built-in operation.

In [15]:
dune = Book("Dune", "Frank Herbert", 412)
hobbit = Book("The Hobbit", "J.R.R. Tolkien", 310)
dune_copy = Book("Dune", "Frank Herbert", 412)

print(dune)              # calls __str__
print(repr(hobbit))      # calls __repr__
print(dune == dune_copy) # calls __eq__ -> True
print(dune == hobbit)    # calls __eq__ -> False

books = [dune, hobbit]
books.sort()             # calls __lt__ to compare
for book in books:
    print(book)

"Dune" by Frank Herbert
Book('The Hobbit', 'J.R.R. Tolkien', 310)
True
False
"The Hobbit" by J.R.R. Tolkien
"Dune" by Frank Herbert


- `print(dune)` calls `dune.__str__()` behind the scenes.
- `repr(hobbit)` calls `hobbit.__repr__()`. It returns a technical string useful for debugging.
- `dune == dune_copy` calls `dune.__eq__(dune_copy)`. They are equal even though they are different objects, because the title and author match.
- `books.sort()` uses `__lt__` to decide the order. Here, books are sorted by page count.

> Special methods turn your custom class into a **first-class citizen** of Python. Your objects behave like built-in types.

<Note type="tip">

Start with `__str__`, `__repr__`, and `__eq__`. They cover most needs. Add more special methods only when your class needs deeper integration with Python syntax (iteration, indexing, addition, etc.).

</Note>

## Resources 📚📚

- [Official Python documentation](https://docs.python.org/3/tutorial/classes.html)
- [Real Python's OOP guides](https://realpython.com/python3-object-oriented-programming/)
- [Python data model](https://docs.python.org/3/reference/datamodel.html) 
- [SOLID principles](https://en.wikipedia.org/wiki/SOLID) 